# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

The lane is a **ranking/scoring problem**, so I use classifier probabilities as ranking scores and evaluate the ordering with **Precision@K**.

I compare two deliberately modest models:

- **Logistic Regression** — readable linear reference with balanced class weights.
- **Random Forest** — the main nonlinear candidate because the decision depends on interactions between visibility, freshness, position, CTR, engagement, and content depth.

The model is not rewarded for complexity. It has to beat the frozen Week-4 rule on the same held-out clients and the same metric.

In [ ]:
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

REPO_URL="https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR="flyrank-ml-internship"

def find_repo_root():
    here=Path.cwd().resolve()
    for candidate in [here,*here.parents]:
        if (candidate/"data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root=find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root=Path(REPO_DIR).resolve()
os.chdir(root)

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["target"]=df["trend_direction"].str.lower().eq("down").astype(int)

features=[
    "impressions_90d","clicks_90d","sessions_90d","avg_position","ctr",
    "content_age_days","days_since_last_update","word_count",
    "engagement_rate","scroll_rate","days_with_impressions","days_with_sessions"
]
print("Rows:",len(df),"Features:",len(features),"Base rate:",round(df["target"].mean(),3))
print("Forbidden target siblings excluded:", set(["trend_direction","trend_pct"]).isdisjoint(features))


## 2. Split design

Validation is **client-grouped**. Whole clients are assigned either to train or test, never both. This is stricter than a random row split because pages from the same client can share hidden structure, editorial patterns, and measurement behavior.

To stay comparable with Week 4, I reproduce the exact frozen split policy: seed 42, 20% of unique clients held out.

This is honest for the current starter proxy because it asks whether the ranking generalizes to clients unseen during training. It is still not the final temporal capstone design; a true future-outcome target should also be evaluated forward in time.

In [ ]:
rng=np.random.default_rng(42)
clients=df["client_id"].drop_duplicates().to_numpy()
test_n=max(1,int(round(len(clients)*0.20)))
test_clients=set(rng.permutation(clients)[:test_n])
test_mask=df["client_id"].isin(test_clients).to_numpy()
train_idx=np.where(~test_mask)[0]
test_idx=np.where(test_mask)[0]

X=df[features]
y=df["target"].to_numpy()

print("Train rows:",len(train_idx),"Test rows:",len(test_idx))
print("Train clients:",df.iloc[train_idx]["client_id"].nunique())
print("Test clients:",df.iloc[test_idx]["client_id"].nunique())
print("Client overlap:",len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])))
print("Test base rate:",round(y[test_idx].mean(),3))
assert len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"]))==0


## 3. Train + compare vs my baseline

The Week-4 rule is recomputed exactly from its frozen feature-only formula. Models train only on the training clients. All methods are compared on the same held-out clients.

I report Precision@20, Precision@50, Precision@100, Average Precision, and ROC-AUC. The operational selection metric remains **Precision@50**.

In [ ]:
def pct_rank(s):
    return pd.Series(s).rank(pct=True,method="average").fillna(0).to_numpy()

visibility=pct_rank(np.log1p(df["impressions_90d"].clip(lower=0)))
freshness=pct_rank(df["days_since_last_update"].fillna(0))
pos=df["avg_position"].fillna(0).to_numpy()
position=((51-np.clip(pos,1,50))/50)*visibility*(pos>0)
depth=(1-pct_rank(df["word_count"].fillna(df["word_count"].median())))*visibility
baseline=np.clip(0.40*visibility+0.30*freshness+0.25*position+0.05*depth,0,1)

def precision_at_k(labels,scores,k):
    order=np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())

def metric_row(name,labels,scores):
    return {
        "method":name,
        "p20":precision_at_k(labels,scores,20),
        "p50":precision_at_k(labels,scores,50),
        "p100":precision_at_k(labels,scores,100),
        "avg_precision":float(average_precision_score(labels,scores)),
        "roc_auc":float(roc_auc_score(labels,scores))
    }

prep=Pipeline([("imputer",SimpleImputer(strategy="median"))])
logit=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scale",StandardScaler()),
    ("model",LogisticRegression(class_weight="balanced",max_iter=1000,random_state=42))
])
rf=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",RandomForestClassifier(
        n_estimators=300,max_depth=10,min_samples_leaf=25,
        class_weight="balanced_subsample",n_jobs=-1,random_state=42
    ))
])

logit.fit(X.iloc[train_idx],y[train_idx])
rf.fit(X.iloc[train_idx],y[train_idx])
logit_score=logit.predict_proba(X.iloc[test_idx])[:,1]
rf_score=rf.predict_proba(X.iloc[test_idx])[:,1]

rows=[
    metric_row("fixed_rule",y[test_idx],baseline[test_idx]),
    metric_row("logistic_regression",y[test_idx],logit_score),
    metric_row("random_forest",y[test_idx],rf_score)
]
result_table=pd.DataFrame(rows).sort_values("p50",ascending=False)
print(result_table.round(3).to_string(index=False))

best_name=result_table.iloc[0]["method"]
print("\nBest by Precision@50:",best_name)
print("Test base rate:",round(y[test_idx].mean(),3))

metrics_payload={
    "split":"client_holdout",
    "seed":42,
    "features":features,
    "test_base_rate":float(y[test_idx].mean()),
    "results":rows,
    "best_by_p50":best_name
}
Path("work/outputs").mkdir(parents=True,exist_ok=True)
Path("work/outputs/w05_model_metrics.json").write_text(json.dumps(metrics_payload,indent=2))


## 4. Errors and interpretation

The goal of the error review is not to excuse misses; it is to understand where the ranking is brittle.

For the random forest I inspect:
- the top feature importances,
- high-scoring false positives,
- low-scoring false negatives,
- and whether any suspicious target sibling appears among the inputs.

A false positive may still be a sensible review candidate — the proxy label is not the same thing as editorial value. A false negative may reflect noisy low-volume behavior or signals the starter snapshot does not contain.

In [ ]:
rf_model=rf.named_steps["model"]
importance=pd.DataFrame({
    "feature":features,
    "importance":rf_model.feature_importances_
}).sort_values("importance",ascending=False)
print("Top features:")
print(importance.head(8).round(4).to_string(index=False))

audit=df.iloc[test_idx][["content_id","client_id","impressions_90d","avg_position","ctr","content_age_days","days_since_last_update"]].copy()
audit["target"]=y[test_idx]
audit["score"]=rf_score
audit["pred"]=(audit["score"]>=0.5).astype(int)

false_pos=audit[(audit["target"]==0)&(audit["pred"]==1)].sort_values("score",ascending=False).head(3)
false_neg=audit[(audit["target"]==1)&(audit["pred"]==0)].sort_values("score").head(3)

print("\nHigh-score false positives:")
print(false_pos.drop(columns=["content_id","client_id"]).round(3).to_string(index=False))
print("\nLow-score false negatives:")
print(false_neg.drop(columns=["content_id","client_id"]).round(3).to_string(index=False))

assert "trend_direction" not in features and "trend_pct" not in features
print("\nLeakage sibling check: PASS")

importance.to_json("work/outputs/w05_feature_importance.json",orient="records",indent=2)


## Self-check

- [x] Method choice matches the ranking/scoring question
- [x] Split is client-grouped and reproduces the frozen Week-4 split
- [x] Baseline and models use the same test rows and Precision@K metrics
- [x] Target-derived fields are excluded from model inputs
- [x] Error cases and feature importance are inspected
- [ ] Notebook executed top to bottom with visible outputs
- [ ] Final metrics receipt committed
- [ ] Submit the public repository URL on the ML-08 card